<a href="https://colab.research.google.com/github/PavloLoiek/ukraine-air-raid-alerts/blob/main/air_raid_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# pandas is the standard Python library for working with tables of data.
import pandas as pd

In [12]:
# --- Cell 2: load the dataset, PINNED to an exact upstream version (reproducible) ---
# We pin to a specific commit of the source repo so the data is FROZEN — anyone
# re-running this notebook gets the identical dataset, even though the live source
# updates daily.
# Source: github.com/Vadimkin/ukrainian-air-raid-sirens-dataset (eTryvoga / volunteer data)
# Pinned commit: f8a2866 (2026-06-24)
url = ("https://raw.githubusercontent.com/Vadimkin/ukrainian-air-raid-sirens-dataset/"
       "f8a2866b97d5379fef5a18d260ecae2081b2489a/datasets/volunteer_data_en.csv")
df = pd.read_csv(url, parse_dates=["started_at", "finished_at"])

print(f"Loaded {len(df):,} alerts (frozen to upstream commit f8a2866, 2026-06-24).")
df.head()

Loaded 101,869 alerts (frozen to upstream commit f8a2866, 2026-06-24).


,region,started_at,finished_at,naive
0,Kyiv City,2022-02-25 16:36:22+00:00,2022-02-25 17:06:22+00:00,True
1,Cherkaska oblast,2022-02-25 18:36:21+00:00,2022-02-25 19:32:11+00:00,False
2,Rivnenska oblast,2022-02-25 18:56:44+00:00,2022-02-25 19:26:44+00:00,True
3,Zaporizka oblast,2022-02-25 18:57:51+00:00,2022-02-25 19:27:51+00:00,True
4,Volynska oblast,2022-02-25 19:41:57+00:00,2022-02-26 04:01:55+00:00,False


In [ ]:
print("Columns:", list(df.columns))
print("Date range:", df["started_at"].min(), "->", df["started_at"].max())
print("Number of regions:", df["region"].nunique())
print()
print("Top 10 regions by number of alerts:")
df["region"].value_counts().head(10)

In [ ]:
# --- Session 2, Step A: prepare time columns ---
# Raw times are UTC. Ukraine runs on Kyiv time (UTC+2 winter / +3 summer),
# so to ask "what HOUR do alerts happen", we convert to local Kyiv time first.
df['start_kyiv'] = df['started_at'].dt.tz_convert('Europe/Kyiv')

# Break the timestamp into pieces we can group by later:
df['date']    = df['start_kyiv'].dt.date            # calendar day
df['hour']    = df['start_kyiv'].dt.hour            # 0-23, local time
df['weekday'] = df['start_kyiv'].dt.day_name()      # Monday..Sunday
df['month']   = df['start_kyiv'].dt.to_period('M')  # e.g. 2022-02

# How long each alert lasted, in minutes (used later; ~5% are estimated):
df['duration_min'] = (df['finished_at'] - df['started_at']).dt.total_seconds() / 60

df[['region','start_kyiv','hour','weekday','month','duration_min','naive']].head()

In [ ]:
# --- Session 2, Step B: how alert frequency changed over the war ---
import matplotlib.pyplot as plt

# "Resampling" is the core time-series move: take individual events and
# count them per time bucket. Here = number of alerts per MONTH, all Ukraine.
monthly = df.set_index('start_kyiv').resample('ME').size()

plt.figure(figsize=(12,4))
monthly.plot()
plt.title('Air-raid alerts per month — all Ukraine (Feb 2022 – Jun 2026)')
plt.ylabel('alerts per month')
plt.xlabel('')
plt.grid(alpha=0.3)
plt.show()

print(monthly.tail(6))

In [ ]:
# --- Session 2, Step C: what time of day do alerts start? (Kyiv local) ---
by_hour = df['hour'].value_counts().sort_index()

plt.figure(figsize=(12,4))
by_hour.plot(kind='bar')
plt.title('Alerts by hour of day (Kyiv local time)')
plt.xlabel('hour of day  (0 = midnight, 12 = noon)')
plt.ylabel('total alerts')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# --- alerts by day of week ---
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
by_weekday = df['weekday'].value_counts().reindex(order)

plt.figure(figsize=(10,4))
by_weekday.plot(kind='bar', color='indianred')
plt.title('Alerts by day of week')
plt.xlabel('')
plt.ylabel('total alerts')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# --- Session 2, Step D: alerts by region — ALL regions ---
by_region = df['region'].value_counts()
print(by_region)

plt.figure(figsize=(10, 9))
by_region.sort_values().plot(kind='barh')   # all regions, smallest at bottom
plt.title('Total air-raid alerts by region — all 25 regions, Feb 2022 – Jun 2026')
plt.xlabel('total alerts')
plt.ylabel('')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- how the most-targeted regions changed over the war ---
top6 = by_region.head(6).index.tolist()

# count alerts per month, split by region (rows = months, columns = regions)
monthly_region = (df[df['region'].isin(top6)]
                  .groupby(['month', 'region']).size()
                  .unstack('region'))

monthly_region.plot(figsize=(13,5))
plt.title('Monthly alerts over time — top 6 most-alerted regions')
plt.ylabel('alerts per month')
plt.xlabel('')
plt.legend(title='region', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 10: ALL 25 regions over time, with legend ---
import matplotlib.pyplot as plt
import numpy as np

# count alerts per month, split by region (rows = months, columns = regions)
all_regions = df.groupby(['month', 'region']).size().unstack('region').fillna(0)

# order regions most→least alerts so the legend reads top-down by size
all_regions = all_regions[df['region'].value_counts().index]

# 25 distinct colors so each line has its own
colors = plt.cm.turbo(np.linspace(0, 1, all_regions.shape[1]))

ax = all_regions.plot(figsize=(15, 8), color=colors, linewidth=1.4)
ax.set_title('Monthly alerts over time — all 25 regions')
ax.set_ylabel('alerts per month')
ax.set_xlabel('')
ax.grid(alpha=0.3)
ax.legend(title='region (most → least alerts)',
          bbox_to_anchor=(1.01, 1), loc='upper left', ncol=1, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- Session 3, Cell 11: install Prophet + build a daily alert count ---
!pip install prophet -q          # Meta's forecasting library; ~30-60s to install

import pandas as pd
import logging
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)   # hush noisy logs

# Prophet always wants exactly two columns: ds = date, y = the thing to forecast.
# We forecast the NUMBER OF ALERTS PER DAY across all Ukraine.
daily = df.set_index('start_kyiv').resample('D').size().rename('y')
daily.index = daily.index.tz_localize(None)     # Prophet requires tz-naive dates

prophet_df = daily.reset_index().rename(columns={'start_kyiv': 'ds'})
prophet_df = prophet_df.iloc[:-1]               # drop today — it's an incomplete day

print(prophet_df.tail())
print('rows:', len(prophet_df),
      '| from', prophet_df['ds'].min().date(), 'to', prophet_df['ds'].max().date())

In [ ]:
# --- Session 3, Cell 12: fit the model and forecast to end of 2026 ---
from prophet import Prophet

m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
m.fit(prophet_df)

# how many days from our last data point to 31 Dec 2026?
periods = (pd.Timestamp('2026-12-31') - prophet_df['ds'].max()).days
future = m.make_future_dataframe(periods=periods)   # extends the timeline forward
forecast = m.predict(future)

fig = m.plot(forecast)
fig.gca().set_title('Daily air-raid alerts — history + projection to end of 2026')
fig.gca().set_ylabel('alerts per day')
plt.show()

In [ ]:
# --- Session 3, Cell 13: split the forecast into trend + seasonality ---
fig2 = m.plot_components(forecast)
plt.show()

In [ ]:
# --- Session 3, Cell 14: the projection in numbers (not eyeballed) ---
eoy = forecast[forecast['ds'] == '2026-12-31'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
print("Projected alerts/day on 31 Dec 2026:")
print(eoy.to_string(index=False))

fc = forecast.set_index('ds')
future_only = fc[fc.index > prophet_df['ds'].max()]
monthly_proj = future_only['yhat'].resample('ME').sum().round(0)
print("\nProjected TOTAL alerts per month (forecast period):")
print(monthly_proj)